# GSM8K benchmark on Anthropic (Eval Protocol)

Bare-bones notebook: call an Anthropic model on GSM8K rows, grade with **regex** or an **LLM judge**, print accuracy.

**Prereqs:** Jupyter kernel = conda `cookbook` env, and `ANTHROPIC_API_KEY` set in your shell environment.

In [1]:
# Run once if imports fail (uses the notebook kernel's Python).
import sys
!{sys.executable} -m pip install -q -e "../../.[eval]"

In [14]:
# --- edit these ---
SONNET_MODEL = "anthropic/claude-sonnet-4-5"
OPUS_MODEL = "anthropic/claude-opus-4-8"
MODEL = SONNET_MODEL  # switch to OPUS_MODEL to compare

JUDGE_MODEL = SONNET_MODEL  # only used when GRADING_MODE="llm_judge"
GRADING_MODE = "regex"  # "regex" or "llm_judge"
MAX_ROWS = None  # smoke test: 2-3 rows. Set to None for the full dataset.

DATASET_URL = "https://raw.githubusercontent.com/eval-protocol/python-sdk/main/development/gsm8k_sample.jsonl"
TEMPERATURE = 0.0
CONCURRENCY = 4

In [15]:
import asyncio
import os
import re

import litellm
from eval_protocol.common_utils import load_jsonl
from eval_protocol.models import EvaluateResult, EvaluationRow
from eval_protocol.pytest import SingleTurnRolloutProcessor
from eval_protocol.pytest.types import RolloutProcessorConfig

if not os.getenv("ANTHROPIC_API_KEY"):
    raise EnvironmentError("Set ANTHROPIC_API_KEY in your shell environment before running.")

# Strict mode: do not silently drop malformed/unsupported LiteLLM params.
litellm.drop_params = False

In [16]:
def load_rows(url: str, max_rows: int | None) -> list[EvaluationRow]:
    raw = load_jsonl(url)
    rows = [EvaluationRow(**item) for item in raw]
    if max_rows is not None:
        rows = rows[:max_rows]
    for i, row in enumerate(rows):
        row.input_metadata.row_id = row.input_metadata.row_id or f"row-{i}"
    return rows


rows = load_rows(DATASET_URL, MAX_ROWS)
print(f"Loaded {len(rows)} rows ({'smoke test' if MAX_ROWS else 'full run'})")

Loaded 1000 rows (full run)


In [17]:
_ANSWER_TAG_RE = re.compile(r"<answer>\s*(.*?)\s*</answer>", re.IGNORECASE | re.DOTALL)
_DIGITS_RE = re.compile(r"(-?\d+(?:\.\d+)?)")


def extract_answer(text: str) -> str | None:
    """Pull the numeric answer from <answer> tags (Eval Protocol GSM8K format)."""
    if not text:
        return None
    m = _ANSWER_TAG_RE.search(text)
    chunk = m.group(1) if m else text
    digits = _DIGITS_RE.findall(chunk.replace(",", ""))
    return digits[-1] if digits else None


def user_question(row: EvaluationRow) -> str:
    for msg in row.messages:
        if msg.role == "user":
            return str(msg.content)
    return ""


def model_response(row: EvaluationRow) -> str:
    return str(row.messages[-1].content)


def grade_regex(row: EvaluationRow) -> EvaluateResult:
    pred = extract_answer(model_response(row))
    gt = extract_answer(str(row.ground_truth))
    if pred is None or gt is None:
        return EvaluateResult(score=0.0, reason=f"missing answer (pred={pred}, gt={gt})")
    ok = pred == gt
    return EvaluateResult(score=1.0 if ok else 0.0, reason=f"pred={pred}, gt={gt}")


async def grade_llm_judge(row: EvaluationRow) -> EvaluateResult:
    prompt = (
        "Grade this math answer. Reply with exactly one word: CORRECT or INCORRECT.\n\n"
        f"Question: {user_question(row)}\n"
        f"Model answer: {model_response(row)}\n"
        f"Reference answer: {row.ground_truth}"
    )
    resp = await litellm.acompletion(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
    )
    verdict = (resp.choices[0].message.content or "").strip().upper()
    ok = "CORRECT" in verdict and "INCORRECT" not in verdict
    return EvaluateResult(score=1.0 if ok else 0.0, reason=verdict)

In [ ]:
async def run_benchmark(rows: list[EvaluationRow]) -> list[EvaluationRow]:
    processor = SingleTurnRolloutProcessor(
        drop_trailing_assistant_messages=True  # True (default) -> do not send final assistant message
    )
    config = RolloutProcessorConfig(
        completion_params={"model": MODEL, "temperature": TEMPERATURE},
        mcp_config_path="",  # MCP config is for agent/tool-calling evals
        semaphore=asyncio.Semaphore(CONCURRENCY),
    )

    rollout_tasks = processor(rows, config)
    completed = await asyncio.gather(*rollout_tasks)

    graded: list[EvaluationRow] = []
    for row in completed:
        if GRADING_MODE == "regex":
            row.evaluation_result = grade_regex(row)
        elif GRADING_MODE == "llm_judge":
            row.evaluation_result = await grade_llm_judge(row)
        else:
            raise ValueError(f"Unknown GRADING_MODE: {GRADING_MODE}")
        graded.append(row)
    return graded


results = await run_benchmark(rows)

In [ ]:
scores = [r.evaluation_result.score for r in results if r.evaluation_result is not None]
accuracy = sum(scores) / len(scores) if scores else 0.0

print(f"Model: {MODEL}")
print(f"Grading: {GRADING_MODE}")
print(f"Rows: {len(results)}")
print(f"Accuracy: {accuracy:.1%} ({int(sum(scores))}/{len(scores)})")
print()

for row in results[:5]:
    q = user_question(row)[:80].replace("\n", " ")
    ans = extract_answer(model_response(row))
    gt = extract_answer(str(row.ground_truth))
    score = row.evaluation_result.score if row.evaluation_result else float("nan")
    print(f"[{score:.0f}] pred={ans} gt={gt} | {q}...")

Model: anthropic/claude-sonnet-4-5
Grading: regex
Rows: 50
Accuracy: 90.0% (45/50)

[1] pred=72 gt=72 | Natalia sold clips to 48 of her friends in April, and then she sold half as many...
[1] pred=10 gt=10 | Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of ba...
[1] pred=5 gt=5 | Betty is saving money for a new wallet which costs $100. Betty has only half of ...
[1] pred=42 gt=42 | Julie is reading a 120-page book. Yesterday, she was able to read 12 pages and t...
[1] pred=624 gt=624 | James writes a 3-page letter to 2 different friends twice a week.  How many page...
